**Experiment 4: Automated ETL Pipeline**
1. Aim

To build an automated ETL (Extract, Transform, Load) pipeline that extracts data from multiple sources, transforms it into a standardized format, and loads the processed data for storage.

This is the same aim already present in your notebook.

2. Theory

ETL (Extract, Transform, Load) is a data processing approach used to collect data from multiple sources, process and standardize the data, and store the final processed data in a target storage system.

ETL consists of three main stages:

Extract

In the extraction stage, data is collected from different sources such as CSV files, databases, or REST APIs.

In this experiment, data is extracted from:

A CSV file containing student information.
A REST API containing user information.

The REST API returns data in JSON format, which is converted into a Pandas DataFrame.

Transform

The transformation stage improves and standardizes the extracted data.

The following operations are performed:

Selecting required columns
Renaming columns
Handling missing values
Removing duplicate records
Merging data from different sources
Standardizing the final dataset
Load

The loading stage stores the processed data in a target storage system.

In this experiment, the final transformed data is loaded into a SQLite database. The processed data is also saved as a CSV file for further use.

In [ ]:
import pandas as pd
import requests
import sqlite3

**EXTRACT**

  Create Student Dataset

In [ ]:
student_data = {
    "userId": [1, 2, 3, 4, 5],
    "Student_Name": ["Rahul", "Priya", "Amit", "Sneha", "Rohan"],
    "Marks": [85, 90, 78, 88, 92]
}

student_df = pd.DataFrame(student_data)

student_df.to_csv("students.csv", index=False)

print("students.csv created successfully!")

display(student_df)

students.csv created successfully!


,userId,Student_Name,Marks
0,1,Rahul,85
1,2,Priya,90
2,3,Amit,78
3,4,Sneha,88
4,5,Rohan,92


Extract Student Data from CSV

In [ ]:
students = pd.read_csv("students.csv")

print("Student CSV data extracted successfully!")

display(students)

Student CSV data extracted successfully!


,userId,Student_Name,Marks
0,1,Rahul,85
1,2,Priya,90
2,3,Amit,78
3,4,Sneha,88
4,5,Rohan,92


Extract Data from REST API

In [ ]:
users_url = "https://jsonplaceholder.typicode.com/users"

response = requests.get(users_url)

print("API Status Code:", response.status_code)

API Status Code: 200


Convert API JSON into DataFrame

In [ ]:
users_data = response.json()

users = pd.DataFrame(users_data)

print("API data extracted successfully!")

display(users.head())

API data extracted successfully!


,id,name,username,email,address,phone,website,company
0,1,Leanne Graham,Bret,Sincere@april.biz,"{'street': 'Kulas Light', 'suite': 'Apt. 556',...",1-770-736-8031 x56442,hildegard.org,"{'name': 'Romaguera-Crona', 'catchPhrase': 'Mu..."
1,2,Ervin Howell,Antonette,Shanna@melissa.tv,"{'street': 'Victor Plains', 'suite': 'Suite 87...",010-692-6593 x09125,anastasia.net,"{'name': 'Deckow-Crist', 'catchPhrase': 'Proac..."
2,3,Clementine Bauch,Samantha,Nathan@yesenia.net,"{'street': 'Douglas Extension', 'suite': 'Suit...",1-463-123-4447,ramiro.info,"{'name': 'Romaguera-Jacobson', 'catchPhrase': ..."
3,4,Patricia Lebsack,Karianne,Julianne.OConner@kory.org,"{'street': 'Hoeger Mall', 'suite': 'Apt. 692',...",493-170-9623 x156,kale.biz,"{'name': 'Robel-Corkery', 'catchPhrase': 'Mult..."
4,5,Chelsey Dietrich,Kamren,Lucio_Hettinger@annie.ca,"{'street': 'Skiles Walks', 'suite': 'Suite 351...",(254)954-1289,demarco.info,"{'name': 'Keebler LLC', 'catchPhrase': 'User-c..."


**TRANSFORM**

Select Required API Columns

In [ ]:
users = users[
    ["id", "name", "email"]
]

display(users.head())

Rename API Columns

In [ ]:
users = users.rename(
    columns={
        "id": "userId",
        "name": "API_Name"
    }
)

print("API columns standardized successfully!")

display(users.head())

API columns standardized successfully!


,userId,API_Name,username,email,address,phone,website,company
0,1,Leanne Graham,Bret,Sincere@april.biz,"{'street': 'Kulas Light', 'suite': 'Apt. 556',...",1-770-736-8031 x56442,hildegard.org,"{'name': 'Romaguera-Crona', 'catchPhrase': 'Mu..."
1,2,Ervin Howell,Antonette,Shanna@melissa.tv,"{'street': 'Victor Plains', 'suite': 'Suite 87...",010-692-6593 x09125,anastasia.net,"{'name': 'Deckow-Crist', 'catchPhrase': 'Proac..."
2,3,Clementine Bauch,Samantha,Nathan@yesenia.net,"{'street': 'Douglas Extension', 'suite': 'Suit...",1-463-123-4447,ramiro.info,"{'name': 'Romaguera-Jacobson', 'catchPhrase': ..."
3,4,Patricia Lebsack,Karianne,Julianne.OConner@kory.org,"{'street': 'Hoeger Mall', 'suite': 'Apt. 692',...",493-170-9623 x156,kale.biz,"{'name': 'Robel-Corkery', 'catchPhrase': 'Mult..."
4,5,Chelsey Dietrich,Kamren,Lucio_Hettinger@annie.ca,"{'street': 'Skiles Walks', 'suite': 'Suite 351...",(254)954-1289,demarco.info,"{'name': 'Keebler LLC', 'catchPhrase': 'User-c..."


Check Missing Values

Check Student Data

In [ ]:
print("Missing values in Student Data:")
print(students.isnull().sum())

Missing values in Student Data:
userId          0
Student_Name    0
Marks           0
dtype: int64


Check API Data

In [ ]:
print("\nMissing values in API Data:")
print(users.isnull().sum())


Missing values in API Data:
userId      0
API_Name    0
username    0
email       0
address     0
phone       0
website     0
company     0
dtype: int64


Handle Missing Values

In [ ]:
students["Student_Name"] = students["Student_Name"].fillna("Unknown")

students["Marks"] = students["Marks"].fillna(
    students["Marks"].mean()
)

users["API_Name"] = users["API_Name"].fillna("Unknown")

users["email"] = users["email"].fillna("Not Available")

print("Missing values handled successfully!")

Missing values handled successfully!


Merge the Two Datasets

In [ ]:
merged_data = pd.merge(
    students,
    users,
    on="userId",
    how="inner"
)

print("Data merged successfully!")

display(merged_data)

Data merged successfully!


,userId,Student_Name,Marks,API_Name,username,email,address,phone,website,company
0,1,Rahul,85,Leanne Graham,Bret,Sincere@april.biz,"{'street': 'Kulas Light', 'suite': 'Apt. 556',...",1-770-736-8031 x56442,hildegard.org,"{'name': 'Romaguera-Crona', 'catchPhrase': 'Mu..."
1,2,Priya,90,Ervin Howell,Antonette,Shanna@melissa.tv,"{'street': 'Victor Plains', 'suite': 'Suite 87...",010-692-6593 x09125,anastasia.net,"{'name': 'Deckow-Crist', 'catchPhrase': 'Proac..."
2,3,Amit,78,Clementine Bauch,Samantha,Nathan@yesenia.net,"{'street': 'Douglas Extension', 'suite': 'Suit...",1-463-123-4447,ramiro.info,"{'name': 'Romaguera-Jacobson', 'catchPhrase': ..."
3,4,Sneha,88,Patricia Lebsack,Karianne,Julianne.OConner@kory.org,"{'street': 'Hoeger Mall', 'suite': 'Apt. 692',...",493-170-9623 x156,kale.biz,"{'name': 'Robel-Corkery', 'catchPhrase': 'Mult..."
4,5,Rohan,92,Chelsey Dietrich,Kamren,Lucio_Hettinger@annie.ca,"{'street': 'Skiles Walks', 'suite': 'Suite 351...",(254)954-1289,demarco.info,"{'name': 'Keebler LLC', 'catchPhrase': 'User-c..."


Validate the Merged Dataset

In [ ]:
print("Merged Dataset:")
display(merged_data)

print("\nShape:", merged_data.shape)

print("\nColumns:", merged_data.columns.tolist())

Merged Dataset:


,userId,Student_Name,Marks,API_Name,username,email,address,phone,website,company
0,1,Rahul,85,Leanne Graham,Bret,Sincere@april.biz,"{'street': 'Kulas Light', 'suite': 'Apt. 556',...",1-770-736-8031 x56442,hildegard.org,"{'name': 'Romaguera-Crona', 'catchPhrase': 'Mu..."
1,2,Priya,90,Ervin Howell,Antonette,Shanna@melissa.tv,"{'street': 'Victor Plains', 'suite': 'Suite 87...",010-692-6593 x09125,anastasia.net,"{'name': 'Deckow-Crist', 'catchPhrase': 'Proac..."
2,3,Amit,78,Clementine Bauch,Samantha,Nathan@yesenia.net,"{'street': 'Douglas Extension', 'suite': 'Suit...",1-463-123-4447,ramiro.info,"{'name': 'Romaguera-Jacobson', 'catchPhrase': ..."
3,4,Sneha,88,Patricia Lebsack,Karianne,Julianne.OConner@kory.org,"{'street': 'Hoeger Mall', 'suite': 'Apt. 692',...",493-170-9623 x156,kale.biz,"{'name': 'Robel-Corkery', 'catchPhrase': 'Mult..."
4,5,Rohan,92,Chelsey Dietrich,Kamren,Lucio_Hettinger@annie.ca,"{'street': 'Skiles Walks', 'suite': 'Suite 351...",(254)954-1289,demarco.info,"{'name': 'Keebler LLC', 'catchPhrase': 'User-c..."



Shape: (5, 10)

Columns: ['userId', 'Student_Name', 'Marks', 'API_Name', 'username', 'email', 'address', 'phone', 'website', 'company']


Handle Missing Values in Merged Data

In [ ]:
merged_data["Student_Name"] = merged_data["Student_Name"].fillna("Unknown")

merged_data["Marks"] = merged_data["Marks"].fillna(
    merged_data["Marks"].mean()
)

merged_data["API_Name"] = merged_data["API_Name"].fillna("Unknown")

merged_data["email"] = merged_data["email"].fillna("Not Available")

print("Missing values handled successfully!")

Missing values handled successfully!


In [ ]:
import requests
import pandas as pd

users_url = "https://jsonplaceholder.typicode.com/users"

response = requests.get(users_url)

print("API Status Code:", response.status_code)

users_data = response.json()

users = pd.DataFrame(users_data)

print("Original API columns:")
print(users.columns.tolist())

API Status Code: 200
Original API columns:
['id', 'name', 'username', 'email', 'address', 'phone', 'website', 'company']


In [ ]:
users = users[["id", "name", "email"]]

users = users.rename(
    columns={
        "id": "userId",
        "name": "API_Name"
    }
)

print("Standardized API data:")
display(users.head())

Standardized API data:


,userId,API_Name,email
0,1,Leanne Graham,Sincere@april.biz
1,2,Ervin Howell,Shanna@melissa.tv
2,3,Clementine Bauch,Nathan@yesenia.net
3,4,Patricia Lebsack,Julianne.OConner@kory.org
4,5,Chelsey Dietrich,Lucio_Hettinger@annie.ca


In [ ]:
print("Student columns:")
print(students.columns.tolist())

display(students)

Student columns:
['userId', 'Student_Name', 'Marks']


,userId,Student_Name,Marks
0,1,Rahul,85
1,2,Priya,90
2,3,Amit,78
3,4,Sneha,88
4,5,Rohan,92


In [ ]:
merged_data = pd.merge(
    students,
    users,
    on="userId",
    how="inner"
)

print("Data merged successfully!")

display(merged_data)

Data merged successfully!


,userId,Student_Name,Marks,API_Name,email
0,1,Rahul,85,Leanne Graham,Sincere@april.biz
1,2,Priya,90,Ervin Howell,Shanna@melissa.tv
2,3,Amit,78,Clementine Bauch,Nathan@yesenia.net
3,4,Sneha,88,Patricia Lebsack,Julianne.OConner@kory.org
4,5,Rohan,92,Chelsey Dietrich,Lucio_Hettinger@annie.ca


In [ ]:
merged_data = merged_data.drop_duplicates()

print("Duplicate records removed successfully!")
print("Duplicate records remaining:", merged_data.duplicated().sum())

Duplicate records removed successfully!
Duplicate records remaining: 0


Final Data Validation

In [ ]:
print("Missing Values:")
print(merged_data.isnull().sum())

print("\nData Types:")
print(merged_data.dtypes)

print("\nDuplicate Records:")
print(merged_data.duplicated().sum())

Missing Values:
userId          0
Student_Name    0
Marks           0
API_Name        0
email           0
dtype: int64

Data Types:
userId           int64
Student_Name    object
Marks            int64
API_Name        object
email           object
dtype: object

Duplicate Records:
0


Create Final Standardized Dataset

In [ ]:
final_data = merged_data[
    ["userId", "Student_Name", "Marks", "API_Name", "email"]
].copy()

print("Final Standardized Dataset:")

display(final_data)

Final Standardized Dataset:


,userId,Student_Name,Marks,API_Name,email
0,1,Rahul,85,Leanne Graham,Sincere@april.biz
1,2,Priya,90,Ervin Howell,Shanna@melissa.tv
2,3,Amit,78,Clementine Bauch,Nathan@yesenia.net
3,4,Sneha,88,Patricia Lebsack,Julianne.OConner@kory.org
4,5,Rohan,92,Chelsey Dietrich,Lucio_Hettinger@annie.ca


In [ ]:
print("Final Dataset Shape:", final_data.shape)

print("\nColumn Names:")
print(final_data.columns.tolist())

print("\nMissing Values:")
print(final_data.isnull().sum())

print("\nDuplicate Records:")
print(final_data.duplicated().sum())

Final Dataset Shape: (5, 5)

Column Names:
['userId', 'Student_Name', 'Marks', 'API_Name', 'email']

Missing Values:
userId          0
Student_Name    0
Marks           0
API_Name        0
email           0
dtype: int64

Duplicate Records:
0


**LOAD** — Create SQLite Database

In [ ]:
connection = sqlite3.connect("ETL_Database.db")

print("SQLite database created successfully!")

SQLite database created successfully!


Load Data into SQLite

In [ ]:
final_data.to_sql(
    "student_information",
    connection,
    if_exists="replace",
    index=False
)

print("Data successfully loaded into SQLite database!")

Data successfully loaded into SQLite database!


Read Data from SQLite

In [ ]:
loaded_data = pd.read_sql(
    "SELECT * FROM student_information",
    connection
)

print("Data retrieved from SQLite database:")

display(loaded_data)

Data retrieved from SQLite database:


,userId,Student_Name,Marks,API_Name,email
0,1,Rahul,85,Leanne Graham,Sincere@april.biz
1,2,Priya,90,Ervin Howell,Shanna@melissa.tv
2,3,Amit,78,Clementine Bauch,Nathan@yesenia.net
3,4,Sneha,88,Patricia Lebsack,Julianne.OConner@kory.org
4,5,Rohan,92,Chelsey Dietrich,Lucio_Hettinger@annie.ca


Perform SQL Query

In [ ]:
result = pd.read_sql(
    "SELECT Student_Name, Marks, email "
    "FROM student_information "
    "WHERE Marks >= 85",
    connection
)

display(result)

,Student_Name,Marks,email
0,Rahul,85,Sincere@april.biz
1,Priya,90,Shanna@melissa.tv
2,Sneha,88,Julianne.OConner@kory.org
3,Rohan,92,Lucio_Hettinger@annie.ca


In [ ]:
final_data.to_csv(
    "ETL_Transformed_Data.csv",
    index=False
)

print("Transformed data saved as CSV successfully!")

Transformed data saved as CSV successfully!


In [ ]:
connection.close()

print("Database connection closed successfully!")

Database connection closed successfully!


Conclusion

The experiment was successfully completed by developing an ETL pipeline that extracted data from multiple sources, including a CSV file and a REST API. The extracted data was transformed by converting it into a standardized tabular format, selecting required attributes, handling missing values, removing duplicate records, and merging the datasets using a common key. Finally, the processed data was loaded into a SQLite database for storage and further analysis and was also exported as a CSV file. Thus, the experiment demonstrated the complete Extract, Transform, and Load workflow using Python.